# 04 — Work–precision and measured performance

Measure achieved error and actual runtime together. In 1D, CN, Strang and Yoshida use the **same FD2 Hamiltonian** and independent matrix-exponential reference. In 2D, split methods use a spectral Hamiltonian and an analytical rotated-oscillator reference up to global phase.

The small legacy benchmark below excludes one warm-up, retains three repetitions, and separates setup, stepping and diagnostics. Small-case timing is noisy and specific to the recorded environment.

The version 2 frozen study later in this notebook uses raw complex L2, one first call and five seeded interleaved repetitions. Its same-H panels and continuum frontier retain separate error budgets.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "wavepacket").is_dir())
print("Python:", sys.executable)

OUT = ROOT / "results" / "notebook_runs" / "04_work_precision"
OUT.mkdir(parents=True, exist_ok=True)
from wavepacket.benchmarks import run_benchmarks

report = run_benchmarks(OUT)
print(report["environment"])
print("Measured comparisons:", len(report["comparisons"]))
for row in report["comparisons"]:
    print(row["family"], row["config"]["solver"]["method"], row["config"]["solver"]["dt"],
          row["state_error"], row["median_seconds"]["stepping_seconds"])


In [ ]:
families = sorted({row["family"] for row in report["comparisons"]})
fig, axes = plt.subplots(1, len(families), figsize=(12, 4), squeeze=False)
for ax, family in zip(axes[0], families, strict=True):
    rows = [row for row in report["comparisons"] if row["family"] == family]
    for method in sorted({row["config"]["solver"]["method"] for row in rows}):
        selected = [row for row in rows if row["config"]["solver"]["method"] == method]
        ax.loglog([row["median_seconds"]["stepping_seconds"] for row in selected],
                  [row["state_error"] for row in selected], "o-", label=method)
    ax.set(xlabel="median stepping time (s)", ylabel="weighted state error", title=family)
    ax.grid(alpha=.2)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / "work_precision.png", dpi=150)
plt.close(fig)
display(Image(filename=str(OUT / "work_precision.png")))


## Review questions

1. Why is the 1D comparison a time-integrator comparison, while equal-grid spectral-versus-FD2 timing would mix two questions?
2. What happens to Yoshida4's work at equal timestep? What happens at equal error?
3. Why can a fast setup matter for a short interactive job and matter less for a long batch?
4. Why are reference generation, plotting and artifact I/O excluded from stepping time?
5. Which part is actually Python work, and which is already compiled numerical-library work?
6. What profiling evidence would justify a native kernel? A wrapper around an FFT is not a new numerical performance result.

Report a target error together with the chosen method, hardware, grid, duration, precision and boundary model. Do not infer a universal speedup from this small benchmark.

## Version 2 frozen accuracy/cost evidence

The release gate requires both canonical studies, their exact current protocols and matching engine hashes. This cell validates those artifacts before reviewing raw state error and measured warm cost. It does not rerun the expensive timed sweep during an unrelated notebook execution. Missing evidence is shown explicitly; stale evidence stops execution.

Review why a method may reach a discrete-H target while its continuum spatial error remains larger. Compare cold setup cost with the warm frontier and inspect every unachieved target.

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from verify_release import inspect_studies
study_path = ROOT / "results/studies/accuracy-cost/results.json"
if not study_path.exists():
    print("Canonical studies absent. Run scripts/run_studies.py accuracy-cost and tunnelling before release.")
else:
    print("Current protocol/source proof:", inspect_studies(ROOT))
    frozen = json.loads(study_path.read_text(encoding="utf-8"))
    fig, axes = plt.subplots(1, len(frozen["panels"]), figsize=(15, 4), squeeze=False)
    for ax, panel in zip(axes[0], frozen["panels"], strict=True):
        for method in dict.fromkeys(row["method"] for row in panel["measurements"]):
            rows = [row for row in panel["measurements"] if row["method"] == method]
            ax.loglog([row["warm_total_median_seconds"] for row in rows],
                      [row["raw_l2_error"] for row in rows], "o-", label=method)
        ax.set(title=panel["id"], xlabel="Warm median total seconds", ylabel="Raw complex L2")
        ax.legend()
        ax.grid(alpha=.2)
        print(panel["id"], "target attainment:", panel["target_attainment"])
    fig.tight_layout()
    fig.savefig(OUT / "frozen_accuracy_cost.png", dpi=150)
    plt.close(fig)
    display(Image(filename=str(OUT / "frozen_accuracy_cost.png")))